In [ ]:
# Week 8 — KNN & Distance Metrics (Euclidean / Manhattan / Cosine)
# Dataset: carclaims (target: FraudFound)

# --- Setup ---
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay

# --- Config ---
DATA_PATH = "/mnt/data/carclaims 12.csv"   # <-- replace if needed
TARGET = "FraudFound"
RANDOM_STATE = 42

# --- Load & basic prep ---
df = pd.read_csv(DATA_PATH)
assert TARGET in df.columns, f"Target '{TARGET}' not in columns."
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# For KNN, high-dim OHE can hurt. Start numeric-only (fast, stable).
X_num = X[num_cols].copy()
X_num = X_num.fillna(X_num.median())

X_tr, X_te, y_tr, y_te = train_test_split(X_num, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

pipe = Pipeline([
    ("sc", StandardScaler()),
    ("clf", KNeighborsClassifier())
])

# Small, fast grid
param = {
    "clf__n_neighbors": [5, 11, 21],
    "clf__metric": ["euclidean", "manhattan", "cosine"],
    "clf__weights": ["uniform", "distance"]
}

gs = GridSearchCV(pipe, param, cv=3, scoring="roc_auc", n_jobs=-1, refit=True)
gs.fit(X_tr, y_tr)

best = gs.best_estimator_
proba = best.predict_proba(X_te)[:,1]
pred  = (proba>=0.5).astype(int)

acc = accuracy_score(y_te, pred)
f1  = f1_score(y_te, pred)
auc = roc_auc_score(y_te, proba)
ap  = average_precision_score(y_te, proba)

print("Best params:", gs.best_params_)
print({"ACC":acc, "F1":f1, "ROC-AUC":auc, "PR-AUC":ap})

# Save a couple plots
Path("wk8_knn_figs").mkdir(exist_ok=True)
# Confusion matrix
disp = ConfusionMatrixDisplay.from_predictions(y_te, pred)
plt.title("Week 8 — KNN Confusion Matrix"); plt.tight_layout()
plt.savefig("wk8_knn_figs/cm_knn.png", dpi=120); plt.close()

# Simple threshold-free PR curve
from sklearn.metrics import precision_recall_curve
prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(); plt.plot(rec,prec); plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"Week 8 — KNN PR Curve (AP={ap:.3f})"); plt.tight_layout()
plt.savefig("wk8_knn_figs/pr_knn.png", dpi=120); plt.close()

# ROC
from sklearn.metrics import roc_curve
fpr,tpr,_=roc_curve(y_te, proba)
plt.figure(); plt.plot(fpr,tpr); plt.plot([0,1],[0,1],'--',lw=1)
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Week 8 — KNN ROC (AUC={auc:.3f})"); plt.tight_layout()
plt.savefig("wk8_knn_figs/roc_knn.png", dpi=120); plt.close()
